# 🌳 İleri Seviye Ağaç Ölçümleri (Kış Dönemi)

Bu notebook, **kışın çekilmiş videolardan** üretilen yapraklardan arındırılmış PLY modellerinden **dal açıları analizi** yapır.

**İş Akışı:**
1. Firestore'dan `advanced_measurements == "pending"` olan kayıtları al
2. PLY dosyasını Firebase Storage'dan indir
3. Dal analizi metrikleri hesapla (DBSCAN kümeleme + PCA)
4. Sonuçları Firestore'a geri yaz

**Hesaplanan Metrikler:**
- Ana dal sayısı
- Ortalama dal açısı (°)
- Min/Max dal açısı (°)
- Dal açısı standart sapması
- Dal simetrisi indeksi (0-1)
- Dallanma yoğunluğu (dal/m)

In [ ]:
# 1. Sistem Bağımlılıkları
!pip install firebase-admin open3d scikit-learn numpy

In [ ]:
# Balyoz 2.0: Open3D'nin bozuk Jupyter başlatıcısını temizle
import os
file_path = '/usr/local/lib/python3.12/dist-packages/open3d/__init__.py'
if os.path.exists(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()
    with open(file_path, 'w') as file:
        for line in lines:
            if 'open3d.visualization.webrtc_server' in line:
                file.write('                pass\n')
            else:
                file.write(line)
    print("Open3D'nin bozuk Jupyter başlatıcısı temizlendi!")

In [ ]:
import os
import numpy as np
import open3d as o3d
import firebase_admin
from firebase_admin import credentials, firestore, storage
from sklearn.cluster import DBSCAN
import urllib.parse
import tempfile

print("Kütüphaneler yüklendi.")

In [ ]:
# --- FIREBASE BAĞLANTISI ---
CREDENTIALS_PATH = "/kaggle/input/datasets/yusuftiryaki/firabase-keys/firebase.json"
STORAGE_BUCKET = "studio-166104040-52130.firebasestorage.app"

if not firebase_admin._apps:
    cred = credentials.Certificate(CREDENTIALS_PATH)
    firebase_admin.initialize_app(cred, {
        'storageBucket': STORAGE_BUCKET
    })

db = firestore.client()
bucket = storage.bucket()

WORKSPACE_DIR = "/kaggle/working/advanced_workspace"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

print("Firebase bağlantısı kuruldu.")

In [ ]:
def download_ply_from_url(ply_url, local_path):
    """
    Firebase Storage'daki PLY dosyasını URL'den indirir.
    URL formatı: https://firebasestorage.googleapis.com/v0/b/{bucket}/o/{path}?alt=media&token={token}
    """
    print(f"[DEBUG] PLY indiriliyor: {local_path}")
    
    # URL'den blob adını çıkar
    parsed_url = urllib.parse.urlparse(ply_url)
    path_segments = parsed_url.path.split('/')
    blob_name_encoded = '/'.join(path_segments[path_segments.index('o') + 1:])
    blob_name = urllib.parse.unquote(blob_name_encoded)
    
    blob = bucket.blob(blob_name)
    blob.download_to_filename(local_path)
    
    file_size_mb = os.path.getsize(local_path) / (1024 * 1024)
    print(f"[SONUÇ] PLY indirildi: {file_size_mb:.1f} MB")
    return local_path

In [ ]:
def load_and_prepare_ply(ply_path, scale_factor=1.0, ground_z_offset=0.0):
    """
    PLY dosyasını yükler, ölçekler, zemine oturtur ve gürültü temizler.
    Firestore'daki transform.scale ve transform.z_offset değerlerini kullanır.
    """
    print(f"[DEBUG] PLY yükleniyor: {ply_path}")
    pcd = o3d.io.read_point_cloud(ply_path)
    print(f"[DEBUG] Ham nokta sayısı: {len(pcd.points)}")
    
    # 1. Gürültü Temizleme
    cl, ind = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    pcd = pcd.select_by_index(ind)
    print(f"[DEBUG] Temizlenmiş nokta sayısı: {len(pcd.points)}")
    
    # 2. Ölçekleme (metre cinsine çevir)
    if scale_factor != 1.0:
        pcd.scale(scale_factor, center=pcd.get_center())
        print(f"[DEBUG] Ölçek çarpanı uygulandı: {scale_factor:.4f}")
    
    # 3. Zemin tespiti ve Z=0'a oturtma
    points = np.asarray(pcd.points)
    ground_z = np.percentile(points[:, 2], 1.0)
    pcd.translate(np.array([0, 0, -ground_z]))
    
    points = np.asarray(pcd.points)
    height = float(np.max(points[:, 2]))
    print(f"[DEBUG] Ağaç boyu: {height:.2f} m")
    
    return pcd, height

In [ ]:
def analyze_branch_structure(pcd, height):
    """
    Kışın yaprakları dökülmüş ağacın 3D nokta bulutundan dal yapısını analiz eder.
    
    Yöntem:
    1. Gövde üstü dallanma bölgesini (Z=0.5m - 1.5m arası) izole et
    2. Gövde çekirdeğini çıkar (merkeze yakın noktalar)
    3. Kalan noktaları DBSCAN ile kümele → her küme = bir dal
    4. Her dalın PCA ana ekseni ile dikey eksen arasındaki açıyı hesapla
    5. Dalların gövde etrafındaki açısal dağılımından simetri hesapla
    
    Returns:
        dict: Tüm dal analizi metrikleri
    """
    points = np.asarray(pcd.points)
    
    # ------------------------------------------------------------------
    # 1. DALLANMA BÖLGESİNİ İZOLE ET
    # Gövdeden ayrılmanın başladığı bölge: topraktan ~0.5m ile ağaç boyunun ~%60'ı arası
    # Bu aralık fıstık ağacının ana dallanma bölgesini yakalar
    # ------------------------------------------------------------------
    branch_zone_min = 0.5  # metre (gövde kısmını atla)
    branch_zone_max = min(height * 0.6, height - 0.3)  # Tepedeki uç noktaları hariç tut
    
    zone_mask = (points[:, 2] > branch_zone_min) & (points[:, 2] < branch_zone_max)
    zone_points = points[zone_mask]
    
    print(f"[DEBUG] Dallanma bölgesi: {branch_zone_min:.1f}m - {branch_zone_max:.1f}m")
    print(f"[DEBUG] Dallanma bölgesindeki nokta sayısı: {len(zone_points)}")
    
    if len(zone_points) < 50:
        print("[UYARI] Dallanma bölgesinde yeterli nokta yok!")
        return _empty_branch_metrics()
    
    # ------------------------------------------------------------------
    # 2. GÖVDE ÇEKİRDEĞİNİ ÇIKAR
    # XY düzleminde merkeze çok yakın noktalar gövdedir, dallar değil
    # Gövde çapının tahmini için alt kısımdaki (0.3-0.5m) noktaları kullan
    # ------------------------------------------------------------------
    trunk_slice = points[(points[:, 2] > 0.3) & (points[:, 2] < 0.5)]
    if len(trunk_slice) > 10:
        trunk_center_xy = np.mean(trunk_slice[:, :2], axis=0)
        trunk_radius = np.median(np.linalg.norm(trunk_slice[:, :2] - trunk_center_xy, axis=1))
    else:
        trunk_center_xy = np.mean(zone_points[:, :2], axis=0)
        trunk_radius = 0.05  # Varsayılan 5cm
    
    print(f"[DEBUG] Gövde merkezi: ({trunk_center_xy[0]:.3f}, {trunk_center_xy[1]:.3f})")
    print(f"[DEBUG] Gövde yarıçapı: {trunk_radius:.3f} m")
    
    # Gövde çekirdeğinin dışındaki noktaları al (dallar)
    distances_to_trunk = np.linalg.norm(zone_points[:, :2] - trunk_center_xy, axis=1)
    branch_mask = distances_to_trunk > (trunk_radius * 1.5)  # Gövde yarıçapının 1.5 katından uzak
    branch_points = zone_points[branch_mask]
    
    print(f"[DEBUG] Gövde çekirdeği çıkarıldı. Kalan dal noktası: {len(branch_points)}")
    
    if len(branch_points) < 30:
        print("[UYARI] Gövde çıkarıldıktan sonra yeterli dal noktası kalmadı!")
        return _empty_branch_metrics()
    
    # ------------------------------------------------------------------
    # 3. DBSCAN KÜMELEMESİ → HER KÜME = BİR DAL
    # eps: Komşuluk mesafesi (metre cinsinden, ağaç boyutuna göre ölçekle)
    # min_samples: Bir kümenin minimum nokta sayısı
    # ------------------------------------------------------------------
    eps = max(0.05, height * 0.02)  # Ağaç boyutuna orantılı
    min_samples = max(10, int(len(branch_points) * 0.01))  # En az %1'lik küme
    
    print(f"[DEBUG] DBSCAN parametreleri: eps={eps:.3f}, min_samples={min_samples}")
    
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(branch_points)
    labels = clustering.labels_
    
    # -1 gürültü noktaları, diğerleri küme/dal indeksleri
    unique_labels = set(labels)
    unique_labels.discard(-1)  # Gürültüyü çıkar
    n_branches = len(unique_labels)
    
    print(f"[SONUÇ] Tespit edilen ana dal sayısı: {n_branches}")
    
    if n_branches == 0:
        print("[UYARI] Hiç dal kümesi bulunamadı!")
        return _empty_branch_metrics()
    
    # ------------------------------------------------------------------
    # 4. HER DAL İÇİN AÇI HESAPLAMA (PCA)
    # Her kümenin (dal) ana yönünü bul, dikeyden sapma açısını hesapla
    # ------------------------------------------------------------------
    branch_angles = []       # Dikey eksenle yapılan açı (derece)
    branch_azimuths = []     # Gövde etrafındaki yön açısı (0-360°)
    branch_lengths = []      # Dal uzunluğu tahmini
    branch_point_counts = [] # Her daldaki nokta sayısı
    
    vertical_axis = np.array([0, 0, 1])  # Dikey referans
    
    for label in unique_labels:
        cluster_points = branch_points[labels == label]
        n_pts = len(cluster_points)
        branch_point_counts.append(n_pts)
        
        if n_pts < 5:
            continue
        
        # --- PCA ile dal yönünü bul ---
        centroid = cluster_points.mean(axis=0)
        centered = cluster_points - centroid
        cov_matrix = np.cov(centered.T)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # En büyük eigenvalue'nun vektörü = dalın ana yönü
        principal_axis = eigenvectors[:, np.argmax(eigenvalues)]
        
        # Dalın yukarı doğru gittiğinden emin ol (Z bileşeni pozitif)
        if principal_axis[2] < 0:
            principal_axis = -principal_axis
        
        # --- Dikey eksenle açı (0° = tam dik, 90° = tam yatay) ---
        cos_angle = np.dot(principal_axis, vertical_axis) / np.linalg.norm(principal_axis)
        angle_from_vertical = float(np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0))))
        branch_angles.append(angle_from_vertical)
        
        # --- Azimut açısı (gövde etrafındaki yön, 0-360°) ---
        # Dal merkezinin gövdeye göre XY düzlemindeki yönü
        dx = centroid[0] - trunk_center_xy[0]
        dy = centroid[1] - trunk_center_xy[1]
        azimuth = float(np.degrees(np.arctan2(dy, dx))) % 360
        branch_azimuths.append(azimuth)
        
        # --- Dal uzunluğu (bounding box köşegeni) ---
        bbox_size = cluster_points.max(axis=0) - cluster_points.min(axis=0)
        branch_length = float(np.linalg.norm(bbox_size))
        branch_lengths.append(branch_length)
        
        print(f"  Dal {label}: {n_pts} nokta, Açı: {angle_from_vertical:.1f}°, "
              f"Azimut: {azimuth:.0f}°, Uzunluk: {branch_length:.2f}m")
    
    if len(branch_angles) == 0:
        return _empty_branch_metrics()
    
    branch_angles = np.array(branch_angles)
    branch_azimuths = np.array(branch_azimuths)
    branch_lengths = np.array(branch_lengths)
    
    # ------------------------------------------------------------------
    # 5. ÖZET METRİKLER
    # ------------------------------------------------------------------
    avg_angle = float(np.mean(branch_angles))
    min_angle = float(np.min(branch_angles))
    max_angle = float(np.max(branch_angles))
    std_angle = float(np.std(branch_angles))
    avg_length = float(np.mean(branch_lengths))
    
    # ------------------------------------------------------------------
    # 6. DAL SİMETRİSİ (Azimut Dağılım Analizi)
    # 360°'yi 4 kadrana böl, her kadrandaki dal sayısını say
    # Mükemmel simetri = her kadranda eşit dal → indeks = 1.0
    # ------------------------------------------------------------------
    n_quadrants = 4
    quadrant_counts = np.zeros(n_quadrants)
    for az in branch_azimuths:
        q = int(az // (360 / n_quadrants)) % n_quadrants
        quadrant_counts[q] += 1
    
    # Simetri: 1 - normalize edilmiş standart sapma
    if np.sum(quadrant_counts) > 0:
        normalized_counts = quadrant_counts / np.sum(quadrant_counts)
        ideal = 1.0 / n_quadrants  # 0.25
        max_std = np.sqrt(ideal * (1 - ideal))  # Olası max sapma
        actual_std = np.std(normalized_counts)
        branch_symmetry = float(1.0 - min(actual_std / max_std, 1.0))
    else:
        branch_symmetry = 0.0
    
    # ------------------------------------------------------------------
    # 7. DALLANMA YOĞUNLUĞU (dal/m)
    # Dallanma bölgesinin yüksekliği başına düşen dal sayısı
    # ------------------------------------------------------------------
    branching_zone_height = branch_zone_max - branch_zone_min
    branching_density = n_branches / branching_zone_height if branching_zone_height > 0 else 0.0
    
    # ------------------------------------------------------------------
    # 8. SAĞLIK DEĞERLENDİRMESİ
    # ------------------------------------------------------------------
    health_notes = []
    if avg_angle < 30:
        health_notes.append("Dallar çok dik - ışık girişi yetersiz olabilir, açma budaması önerilebilir")
    elif avg_angle > 60:
        health_notes.append("Dallar çok yatık - kırılma riski var, destek/bağlama önerilebilir")
    else:
        health_notes.append("Dal açıları ideal aralıkta (35-55°)")
    
    if branch_symmetry < 0.5:
        health_notes.append("Dal dağılımı asimetrik - tek taraflı budama gerekebilir")
    
    if branching_density > 8:
        health_notes.append("Dallanma yoğunluğu yüksek - seyreltme budaması önerilebilir")
    
    # ------------------------------------------------------------------
    # SONUÇ
    # ------------------------------------------------------------------
    results = {
        "branch_count": n_branches,
        "avg_branch_angle_deg": round(avg_angle, 1),
        "min_branch_angle_deg": round(min_angle, 1),
        "max_branch_angle_deg": round(max_angle, 1),
        "branch_angle_std_deg": round(std_angle, 1),
        "avg_branch_length_m": round(avg_length, 2),
        "branch_symmetry_index": round(branch_symmetry, 3),
        "branching_density_per_m": round(branching_density, 1),
        "health_notes": health_notes,
        "quadrant_distribution": {
            "N": int(quadrant_counts[0]),
            "E": int(quadrant_counts[1]),
            "S": int(quadrant_counts[2]),
            "W": int(quadrant_counts[3])
        },
        "branch_details": [
            {
                "angle_deg": round(float(branch_angles[i]), 1),
                "azimuth_deg": round(float(branch_azimuths[i]), 0),
                "length_m": round(float(branch_lengths[i]), 2)
            }
            for i in range(len(branch_angles))
        ]
    }
    
    print(f"\n{'='*60}")
    print(f"📊 DAL ANALİZİ SONUÇLARI")
    print(f"{'='*60}")
    print(f"  Ana Dal Sayısı:        {n_branches}")
    print(f"  Ortalama Dal Açısı:    {avg_angle:.1f}°")
    print(f"  Min/Max Dal Açısı:     {min_angle:.1f}° / {max_angle:.1f}°")
    print(f"  Dal Açısı Std Sapma:   {std_angle:.1f}°")
    print(f"  Ortalama Dal Uzunluğu: {avg_length:.2f} m")
    print(f"  Dal Simetrisi:         {branch_symmetry:.3f} (1=mükemmel)")
    print(f"  Dallanma Yoğunluğu:    {branching_density:.1f} dal/m")
    print(f"  Kadran Dağılımı:       K={int(quadrant_counts[0])} D={int(quadrant_counts[1])} "
          f"G={int(quadrant_counts[2])} B={int(quadrant_counts[3])}")
    print(f"{'='*60}")
    for note in health_notes:
        print(f"  💡 {note}")
    print(f"{'='*60}")
    
    return results


def _empty_branch_metrics():
    """Dal analizi yapılamadığında döndürülen boş metrik seti."""
    return {
        "branch_count": 0,
        "avg_branch_angle_deg": 0.0,
        "min_branch_angle_deg": 0.0,
        "max_branch_angle_deg": 0.0,
        "branch_angle_std_deg": 0.0,
        "avg_branch_length_m": 0.0,
        "branch_symmetry_index": 0.0,
        "branching_density_per_m": 0.0,
        "health_notes": ["Dal analizi yapılamadı - yeterli veri yok"],
        "quadrant_distribution": {"N": 0, "E": 0, "S": 0, "W": 0},
        "branch_details": []
    }

In [ ]:
def update_firestore_with_advanced_metrics(doc_id, branch_metrics):
    """
    Dal analizi sonuçlarını Firestore'a yazar.
    advanced_measurements alanını 'completed' olarak günceller.
    """
    print(f"[DEBUG] Firestore güncelleniyor: {doc_id}")
    
    # branch_details listesindeki her elemanı Firestore uyumlu dict'e çevir
    branch_details_clean = []
    for detail in branch_metrics.get("branch_details", []):
        branch_details_clean.append({
            "angle_deg": detail["angle_deg"],
            "azimuth_deg": detail["azimuth_deg"],
            "length_m": detail["length_m"]
        })
    
    update_data = {
        "advanced_measurements": "completed",
        "branch_analysis": {
            "branch_count": branch_metrics["branch_count"],
            "avg_branch_angle_deg": branch_metrics["avg_branch_angle_deg"],
            "min_branch_angle_deg": branch_metrics["min_branch_angle_deg"],
            "max_branch_angle_deg": branch_metrics["max_branch_angle_deg"],
            "branch_angle_std_deg": branch_metrics["branch_angle_std_deg"],
            "avg_branch_length_m": branch_metrics["avg_branch_length_m"],
            "branch_symmetry_index": branch_metrics["branch_symmetry_index"],
            "branching_density_per_m": branch_metrics["branching_density_per_m"],
            "health_notes": branch_metrics["health_notes"],
            "quadrant_distribution": branch_metrics["quadrant_distribution"],
            "branch_details": branch_details_clean
        },
        "advanced_measurements_updated": firestore.SERVER_TIMESTAMP
    }
    
    db.collection("trees").document(doc_id).set(update_data, merge=True)
    print(f"[SONUÇ] {doc_id} dal analizi Firestore'a yazıldı!")

In [ ]:
def process_pending_advanced_measurements():
    """
    Firestore'da advanced_measurements == 'pending' olan tüm ağaçları bulur,
    PLY dosyasını indirir, dal analizini yapar ve sonuçları geri yazar.
    """
    print("=" * 60)
    print("🔍 Firebase'de ileri ölçüm bekleyen ağaçlar aranıyor...")
    print("=" * 60)
    
    pending_docs = db.collection('trees').where(
        'advanced_measurements', '==', 'pending'
    ).stream()
    
    processed = 0
    failed = 0
    
    for doc in pending_docs:
        doc_id = doc.id
        tree_data = doc.to_dict()
        
        print(f"\n{'─'*60}")
        print(f"🌳 İşleniyor: {doc_id}")
        print(f"{'─'*60}")
        
        # Durumu 'processing' yap (çakışmayı önlemek için)
        db.collection('trees').document(doc_id).update({
            'advanced_measurements': 'processing'
        })
        
        try:
            # PLY URL kontrolü
            ply_url = tree_data.get('ply_url')
            if not ply_url:
                raise Exception("Bu ağacın PLY dosyası henüz yüklenmemiş (ply_url boş)!")
            
            # Ölçek ve zemin offset bilgilerini al
            transform = tree_data.get('transform', {})
            scale_factor = transform.get('scale', 1.0)
            z_offset = transform.get('z_offset', 0.0)
            
            print(f"[DEBUG] PLY URL: {ply_url[:80]}...")
            print(f"[DEBUG] Ölçek: {scale_factor}, Z Offset: {z_offset}")
            
            # 1. PLY dosyasını indir
            local_ply_path = os.path.join(WORKSPACE_DIR, f"{doc_id}.ply")
            download_ply_from_url(ply_url, local_ply_path)
            
            # 2. PLY'yi yükle ve hazırla
            pcd, height = load_and_prepare_ply(
                local_ply_path, 
                scale_factor=scale_factor
            )
            
            # 3. Dal yapısı analizi
            branch_metrics = analyze_branch_structure(pcd, height)
            
            # 4. Sonuçları Firestore'a yaz
            update_firestore_with_advanced_metrics(doc_id, branch_metrics)
            
            # 5. Geçici dosyayı temizle
            if os.path.exists(local_ply_path):
                os.remove(local_ply_path)
            
            processed += 1
            print(f"\n✅ {doc_id} başarıyla işlendi!")
            
        except Exception as e:
            failed += 1
            print(f"\n❌ HATA: {doc_id} işlenirken hata oluştu: {str(e)}")
            db.collection('trees').document(doc_id).update({
                'advanced_measurements': 'pending',
                'advanced_measurements_error': str(e)
            })
    
    print(f"\n{'='*60}")
    print(f"📋 ÖZET: {processed} başarılı, {failed} başarısız")
    if processed == 0 and failed == 0:
        print("   Bekleyen ileri ölçüm kaydı bulunamadı.")
    print(f"{'='*60}")

In [ ]:
# --- İŞÇİYİ BAŞLAT ---
if __name__ == "__main__":
    process_pending_advanced_measurements()